# A2A Protocol: Comunicación entre Agentes (Google 2025)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/agentes-avanzados/06-a2a-protocol.ipynb)

El protocolo A2A (Agent-to-Agent) de Google permite que agentes de diferentes frameworks se comuniquen con un protocolo estandarizado.

In [ ]:
!pip install fastapi uvicorn anthropic httpx pydantic -q

## 1. Conceptos del protocolo A2A

In [ ]:
from pydantic import BaseModel
from typing import Optional, Any
from datetime import datetime

# Modelos del protocolo A2A
class AgentCard(BaseModel):
    """Describe las capacidades de un agente."""
    name: str
    description: str
    url: str
    version: str = '1.0'
    skills: list[str]
    input_modes: list[str] = ['text']
    output_modes: list[str] = ['text']

class Task(BaseModel):
    """Tarea enviada a un agente."""
    id: str
    session_id: Optional[str] = None
    message: dict  # {role: str, parts: [{type: 'text', text: str}]}
    metadata: dict = {}

class TaskStatus(BaseModel):
    """Estado de una tarea."""
    state: str  # 'submitted', 'working', 'completed', 'failed'
    message: Optional[dict] = None
    timestamp: str = datetime.now().isoformat()

# Ejemplo de AgentCard para un agente Claude
card_agente_claude = AgentCard(
    name='ClaudeAnalysisAgent',
    description='Agente especializado en análisis de texto y documentos con Claude',
    url='http://localhost:8000',
    skills=['text_analysis', 'summarization', 'classification', 'qa'],
    input_modes=['text'],
    output_modes=['text', 'json'],
)

print('AgentCard:')
print(card_agente_claude.model_dump_json(indent=2))

## 2. Servidor A2A con FastAPI

In [ ]:
from fastapi import FastAPI
import anthropic
import uuid

app = FastAPI(title='A2A Agent Server')
client = anthropic.Anthropic()

# Almacén de tareas en memoria (en producción: Redis o BD)
tareas_store: dict[str, dict] = {}

@app.get('/.well-known/agent.json')
def get_agent_card():
    """Endpoint requerido por A2A: describe las capacidades del agente."""
    return card_agente_claude.model_dump()

@app.post('/tasks/send')
async def send_task(tarea: Task):
    """Recibe una tarea y la procesa con Claude."""
    texto = tarea.message['parts'][0]['text']
    
    # Procesar con Claude
    response = client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=1024,
        messages=[{'role': 'user', 'content': texto}],
    )
    
    respuesta_texto = response.content[0].text
    
    status = TaskStatus(
        state='completed',
        message={'role': 'agent', 'parts': [{'type': 'text', 'text': respuesta_texto}]},
    )
    tareas_store[tarea.id] = status.model_dump()
    return status.model_dump()

@app.get('/tasks/{task_id}')
def get_task(task_id: str):
    if task_id not in tareas_store:
        return {'error': 'Tarea no encontrada'}
    return tareas_store[task_id]

print('Servidor A2A definido. Para iniciar: uvicorn main:app --reload')
print('Endpoints:')
print('  GET  /.well-known/agent.json  → AgentCard')
print('  POST /tasks/send              → Enviar tarea')
print('  GET  /tasks/{id}              → Estado de tarea')

## 3. Cliente A2A: enviar tareas a un agente

In [ ]:
import httpx
import uuid

class A2AClient:
    def __init__(self, base_url: str):
        self.base_url = base_url.rstrip('/')

    def discover(self) -> dict:
        """Descubre las capacidades del agente remoto."""
        response = httpx.get(f'{self.base_url}/.well-known/agent.json')
        return response.json()

    def send_task(self, texto: str, session_id: str | None = None) -> dict:
        """Envía una tarea de texto al agente."""
        tarea = {
            'id': str(uuid.uuid4()),
            'session_id': session_id or str(uuid.uuid4()),
            'message': {
                'role': 'user',
                'parts': [{'type': 'text', 'text': texto}],
            },
        }
        response = httpx.post(f'{self.base_url}/tasks/send', json=tarea)
        return response.json()

# Ejemplo de uso (requiere servidor corriendo)
# cliente = A2AClient('http://localhost:8000')
# card = cliente.discover()
# print('Agente descubierto:', card['name'], '-', card['description'])
# resultado = cliente.send_task('Resume en 2 frases qué es el protocolo A2A')
# print('Resultado:', resultado)

print('Cliente A2A listo. Inicia el servidor antes de usar.')
print('\nFlujo A2A:')
print('  1. Descubrir agente (GET /.well-known/agent.json)')
print('  2. Enviar tarea (POST /tasks/send)')
print('  3. Obtener resultado (GET /tasks/{id})')

## Resumen del protocolo A2A

| Componente | Descripción |
|------------|-------------|
| AgentCard | Describe capacidades del agente (skills, modos I/O) |
| Task | Unidad de trabajo enviada entre agentes |
| TaskStatus | Estado: submitted → working → completed/failed |
| `/.well-known/agent.json` | Endpoint estándar de descubrimiento |
| `/tasks/send` | Endpoint para enviar tareas |

A2A hace que agentes de diferentes frameworks (Claude, LangChain, AutoGen) puedan colaborar.